## 6. Area-adjusted SC extent

The adjusted area of class $j$ is (Olofsson et al. 2014, Eq. 10):

$$
\hat{A}_j = A_{\text{total}} \cdot \hat{p}_{+j}
$$

where $\hat{p}_{+j}$ is the column sum of the weighted error matrix $\mathbf{W}$.


In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
# Update these two paths before running.
VAL_POINTS_PATH  = Path('data/validation/val_points_strata.gpkg')
STRATA_RASTER    = Path('data/validation/validation_strata.tif')

# ── Constants ──────────────────────────────────────────────────────────────
CLASSES      = [0, 1, 2, 3, 4]
CLASS_NAMES  = {0: 'Forest', 1: 'Shifting cult.', 2: 'Sec. veg./agr.',
                3: 'Agriculture', 4: 'Mosaic'}
SC_CLASS     = 1        # class index for shifting cultivation
PIXEL_AREA   = 1.0      # km² per pixel (1 km resolution map)
N_STRATA     = 4        # number of sampling strata
N_PER_STRATA = 100      # samples per stratum (equal allocation)

## 1. Load validation points

In [ ]:
df = gpd.read_file(VAL_POINTS_PATH)

# Rename columns to standard names used throughout this notebook
df = df.rename(columns={'class': 'reference', 'pred': 'predicted'})

print(f"Validation points loaded: {len(df)}")
print(f"Strata: {sorted(df['stratum'].unique())}")
print(f"Reference classes: {sorted(df['reference'].unique())}")
print(f"Predicted classes: {sorted(df['predicted'].unique())}")
df.head()

## 2. Stratum pixel counts → area weights

The stratum area weights $W_h = N_h / N$ (proportion of total pixels in each stratum)
are required to compute the area-adjusted error matrix.
Here $N_h$ is the pixel count of stratum $h$ and $N$ is the total pixel count.

In [ ]:
with rasterio.open(STRATA_RASTER) as src:
    strata_raster = src.read(1)

unique, counts = np.unique(strata_raster, return_counts=True)
pixel_counts = {int(v): int(c) for v, c in zip(unique, counts) if int(v) in range(1, 5)}
total_pixels = sum(pixel_counts.values())

# Stratum weights (Wi = Nh / N)
W_h = {h: pixel_counts[h] / total_pixels for h in range(1, N_STRATA + 1)}

print("Stratum pixel counts and weights:")
for h in range(1, N_STRATA + 1):
    print(f"  Stratum {h}: {pixel_counts[h]:>12,} pixels  (W_h = {W_h[h]:.4f})")
print(f"  Total:    {total_pixels:>12,} pixels")

## 3. Mapped area per class

Mapped (uncorrected) area per class from the model prediction raster,
used as the basis for area adjustment.

In [ ]:
# Mapped area in km² (from model prediction raster pixel counts)
# These values come from counting pixels of each class in the prediction raster.
# Replace with values recomputed from your prediction raster if needed.
map_area_km2 = {
    0: 9_465_050.25,   # Forest
    1: 1_776_862.50,   # Shifting cultivation
    2: 10_357_143.25,  # Secondary vegetation / agriculture
    3: 9_827_154.25,   # Agriculture
    4: 4_337_030.00,   # Mosaic
}
total_area_km2 = sum(map_area_km2.values())

print("Mapped area per class:")
for c in CLASSES:
    mha = map_area_km2[c] / 10_000
    pct = map_area_km2[c] / total_area_km2 * 100
    print(f"  Class {c} ({CLASS_NAMES[c]:<22}): {mha:7.2f} Mha  ({pct:.1f}%)")
print(f"  Total: {total_area_km2/10_000:.2f} Mha")

## 4. Area-weighted error matrix (Olofsson et al. 2014, Eq. 9)

The proportional error matrix $\mathbf{W}$ is constructed from per-stratum
sample counts:

$$
\hat{p}_{ij} = W_h \cdot \frac{n_{hij}}{n_h}
$$

where $W_h$ is the stratum weight, $n_{hij}$ is the count of points in stratum $h$
with map class $i$ and reference class $j$, and $n_h$ is the total sample size
in stratum $h$.

In [ ]:
# Build per-stratum confusion matrices
p_h = {}  # p_h[stratum] = proportional confusion matrix for that stratum
for h in range(1, N_STRATA + 1):
    stratum_df = df[df['stratum'] == h]
    n_h = len(stratum_df)

    # Raw counts: rows = predicted class, columns = reference class
    nij = (stratum_df
           .groupby(['predicted', 'reference'])
           .size()
           .unstack(fill_value=0)
           .reindex(index=CLASSES, columns=CLASSES, fill_value=0))

    print(f"\nStratum {h} — raw counts (n={n_h}):")
    print(nij.to_string())

    # Proportional matrix: divide by stratum sample size, weight by W_h
    p_h[h] = nij / n_h * W_h[h]

# Sum across strata to get the overall area-weighted error matrix
W_matrix = sum(p_h.values())

print("\nArea-weighted error matrix (W):")
print(W_matrix.round(4).to_string())

## 5. Area-adjusted accuracy metrics

Following Olofsson et al. (2014) Eqs. 2–4:

$$
\text{UA}_i = \hat{p}_{ii} / \hat{p}_{i+} \qquad
\text{PA}_i = \hat{p}_{ii} / \hat{p}_{+i} \qquad
\text{OA} = \sum_i \hat{p}_{ii}
$$

In [ ]:
UA = {c: W_matrix.loc[c, c] / W_matrix.loc[c].sum()  for c in CLASSES}
PA = {c: W_matrix.loc[c, c] / W_matrix[c].sum()       for c in CLASSES}
OA = sum(W_matrix.loc[c, c] for c in CLASSES)

print("Area-adjusted accuracy metrics (Olofsson et al. 2014)")
print("-" * 55)
print(f"{'Class':<5} {'Name':<26} {'UA':>6} {'PA':>6}")
print("-" * 55)
for c in CLASSES:
    print(f"{c:<5} {CLASS_NAMES[c]:<26} {UA[c]:>6.3f} {PA[c]:>6.3f}")
print("-" * 55)
print(f"Overall Accuracy (OA): {OA:.3f}")

## 6. Area-adjusted SC extent

The adjusted area of class $j$ is (Olofsson et al. 2014, Eq. 10):

$$
\hat{A}_j = A_{\text{total}} \cdot \hat{p}_{+j}
$$

where $\hat{p}_{+j}$ is the column sum of the weighted error matrix $\mathbf{W}$.


In [ ]:
# Proportion of total area attributed to each class (column sums of W)
p_col = {c: W_matrix[c].sum() for c in CLASSES}

# Adjusted area estimates (km²)
A_hat = {c: total_area_km2 * p_col[c] for c in CLASSES}

print("Area-adjusted class extents (Olofsson et al. 2014)")
print("-" * 56)
print(f"{'Class':<5} {'Name':<26} {'Area (Mha)':>12} {'% of total':>12}")
print("-" * 56)
for c in CLASSES:
    a   = A_hat[c] / 10_000
    pct = A_hat[c] / total_area_km2 * 100
    print(f"{c:<5} {CLASS_NAMES[c]:<26} {a:>12.2f} {pct:>11.1f}%")
print("-" * 56)
print(f"\nShifting cultivation (class 1): {A_hat[SC_CLASS]/10_000:.2f} Mha")


## 7. Confusion matrix figure

In [ ]:
# Raw count confusion matrix (rows = predicted, columns = reference)
raw_cm = (df.groupby(['predicted', 'reference'])
            .size()
            .unstack(fill_value=0)
            .reindex(index=CLASSES, columns=CLASSES, fill_value=0))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

labels = [CLASS_NAMES[c] for c in CLASSES]

# Left: raw counts
sns.heatmap(raw_cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=axes[0])
axes[0].set_title('Confusion matrix (counts)', fontsize=12)
axes[0].set_xlabel('Reference class')
axes[0].set_ylabel('Predicted class')
plt.setp(axes[0].get_xticklabels(), rotation=30, ha='right')

# Right: area-weighted proportions
sns.heatmap(W_matrix.round(4), annot=True, fmt='.3f', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=axes[1])
axes[1].set_title('Area-weighted error matrix (Olofsson et al. 2014)', fontsize=12)
axes[1].set_xlabel('Reference class')
axes[1].set_ylabel('Predicted class')
plt.setp(axes[1].get_xticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.savefig('figures/confusion_matrix.pdf', bbox_inches='tight', dpi=300)
plt.show()
print("Saved: figures/confusion_matrix.pdf")

## 8. Summary table for paper

In [ ]:
summary = pd.DataFrame({
    'Class':                 [CLASS_NAMES[c] for c in CLASSES],
    'Mapped area (Mha)':     [round(map_area_km2[c] / 10_000, 2) for c in CLASSES],
    'Adjusted area (Mha)':   [round(A_hat[c] / 10_000, 2) for c in CLASSES],
    'User accuracy':         [f"{UA[c]:.3f}" for c in CLASSES],
    'Producer accuracy':     [f"{PA[c]:.3f}" for c in CLASSES],
})
summary = summary.set_index('Class')

print(f"Overall accuracy (area-adjusted): {OA:.3f}\n")
print(summary.to_string())

summary.to_csv('results/accuracy_summary.csv')
print("\nSaved: results/accuracy_summary.csv")
